# Projeto Prático: Machine Learning & Inteligência de Mercado
## Análise Estratégica da Concentração no Comércio Global de Bens Criativos (Dataset OpenFCS)

---

> **Componente Curricular:** Machine Learning aplicado à Administração
> **Instituição:** Curso de Graduação em Administração
> **Objetivo:** Aplicação prática de Ciência de Dados, Machine Learning e Inteligência Artificial Generativa para diagnosticar padrões de concentração e (re)configuração competitiva no comércio mundial de bens criativos, a partir do acervo aberto OpenFCS (UNCTAD, alinhado ao UNESCO Framework for Cultural Statistics 2025).

---

### Corpo Docente & Contato

| Atributo | Detalhes |
| :--- | :--- |
| **Professor** | **Sérgio Assunção Monteiro, D.Sc.** |
| **Conecte-se no LinkedIn** | [🌐 linkedin.com/in/sergio-assunção-monteiro](https://www.linkedin.com/in/sergio-assun%C3%A7%C3%A3o-monteiro-b781897b/) |
| **Currículo Lattes** | [🔬 lattes.cnpq.br/9489191035734025](http://lattes.cnpq.br/9489191035734025) |
| **Repositório GitHub** | [💻 github.com/sergiomonteiro76](https://github.com/sergiomonteiro76) |

---

### Sobre este Notebook
Este ambiente foi configurado para que os alunos atuem como **Analistas de Inteligência de Mercado**. Ao longo do semestre, com apoio de modelos de linguagem (IA) integrados ao ecossistema do Google Colab, vamos reconstruir — do dado bruto ao modelo preditivo — o diagnóstico de estrutura competitiva de um setor econômico real: o comércio internacional de bens criativos.

* **Fonte de dados:** [OpenFCS Dataset](https://doi.org/10.5281/zenodo.21211053) — Monteiro & Dubeux (2026), CC-BY-4.0.
* **Material de apoio:** Capítulos 1 a 3 das notas de aula (Ambiente e primeiro contato; Python/pandas/NumPy; Bases de Dados e SQL).

---
# 📝 AVALIAÇÃO 1 (A1) — Machine Learning Aplicado à Administração

| Atributo | Detalhes |
| :--- | :--- |
| **Valor** | 10,0 pontos |
| **Conteúdo cobrado** | Capítulos 1 a 3 (ambiente e primeiro contato; Python, pandas e NumPy; Bases de Dados e SQL) |
| **Formato** | Individual ou em dupla, conforme orientação do professor |
| **Consulta** | Material de aula, internet e uso de IA são permitidos e incentivados — **desde que todo uso de IA seja declarado** (ver regra abaixo) |
| **Entrega** | Repositório público no GitHub, com README, + link enviado no ambiente virtual indicado pelo professor |
| **Prazo** | *(a definir pelo professor — inserir data e horário aqui)* |

## Regras gerais

1. **Não altere a Parte 1** (leitura dos dados). Ela já está pronta e testada — resolva as questões a partir dela.
2. Cada questão tem uma célula de código reservada para a resposta. Adicione quantas células precisar logo abaixo dela, mas **não apague os enunciados**.
3. Onde houver uma pergunta de interpretação (🗣️), responda **em texto, na célula de markdown indicada** — código sozinho não vale a pontuação da interpretação.
4. Se você usar uma IA (Claude, ChatGPT, Gemini etc.) para gerar ou revisar parte do código, **declare isso brevemente** na célula de resposta (ex.: *"Consultei uma IA para revisar a sintaxe do JOIN"*). Não é motivo de desconto — é prática profissional esperada. O que não é aceito é entregar uma resposta que você não entende e não consegue explicar se perguntado.
5. Todas as consultas SQL devem usar `duckdb.sql(...)`, exatamente como praticado em aula.

## Critérios de entrega (obrigatórios — leia antes de começar)

1. Ao terminar, baixe este notebook do Colab: **Arquivo → Fazer download → Download .ipynb**.
2. Crie um repositório público no GitHub (ex.: `ml-administracao-a1-seu-nome`).
3. Suba o notebook (`.ipynb`) para o repositório.
4. Crie um arquivo `README.md` na raiz do repositório (use o modelo fornecido pelo professor).
5. Copie o link do repositório e envie no local indicado pelo professor, dentro do prazo.

> ⚠️ **A entrega via GitHub com README é condição obrigatória para a correção.** Notebooks enviados por e-mail, sem repositório público ou sem README, **não serão corrigidos** até regularização dentro do prazo estipulado pelo professor.

---
## Parte 1 — Leitura dos Dados (fornecida pelo professor)

Esta parte já está pronta. **Apenas execute as células abaixo, na ordem.** Ela baixa o acervo OpenFCS, aplica o filtro de resolução de sete domínios (visto no Capítulo 2) e carrega as três tabelas que vocês vão usar: `edges7` (fato), `entities` e `crosswalk` (dimensões).

In [21]:
# --- Download e extração do acervo (Capítulo 1) ---
import requests, zipfile, io, os
import pandas as pd

url = "https://zenodo.org/records/21211053/files/openfcs_v1.0.0.zip?download=1"
resp = requests.get(url)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall("openfcs")

endereco = "openfcs/openfcs-1.0.0/data/derived/"
print("Download e extracao concluidos.")

Download e extracao concluidos.


In [22]:
# --- Carregamento das tabelas (Capítulos 1 e 3) ---
edges = pd.read_csv(endereco + "trade_edges.csv")
entities = pd.read_csv(endereco + "entities.csv")
crosswalk = pd.read_csv(endereco + "products_crosswalk.csv")

print(f"trade_edges.csv:        {edges.shape[0]:,} linhas")
print(f"entities.csv:           {entities.shape[0]:,} linhas")
print(f"products_crosswalk.csv: {crosswalk.shape[0]:,} linhas")

trade_edges.csv:        2,197,978 linhas
entities.csv:           309 linhas
products_crosswalk.csv: 14 linhas


In [23]:
# --- Filtro de resolucao de sete dominios (Capítulo 2) ---
print(edges["resolution"].value_counts())

edges7 = edges[edges["resolution"] == "cer7"].copy()
print(f"\nTabela fato (edges7): {len(edges7):,} linhas")   # deve dar 1.026.400

resolution
craft_sub    1171578
cer7         1026400
Name: count, dtype: int64

Tabela fato (edges7): 1,026,400 linhas


In [24]:
# --- DuckDB (Capítulo 3) ---
!pip install duckdb --quiet
import duckdb

print("Ambiente pronto. edges7, entities e crosswalk estao carregados.")

Ambiente pronto. edges7, entities e crosswalk estao carregados.


### ✅ Checkpoint
Antes de continuar, confirme:
- `edges7` tem **1.026.400** linhas;
- `edges7.columns` inclui `economy`, `partner`, `year`, `fcs_domain`, `value_usd_millions`, `cer_code`;
- `duckdb.sql("SELECT * FROM edges7 LIMIT 3").df()` roda sem erro.

Se algo não bater, **não prossiga** — revise a Parte 1 antes de ir para as questões.

In [25]:
# Celula de checkpoint - rode e confira
print(edges7.shape)
print(edges7.columns.tolist())
duckdb.sql("SELECT * FROM edges7 LIMIT 3").df()

(1026400, 11)
['edge_id', 'economy', 'partner', 'product', 'year', 'value_usd_millions', 'flow', 'resolution', 'cer_code', 'fcs_domain', 'mapping_status']


,edge_id,economy,partner,product,year,value_usd_millions,flow,resolution,cer_code,fcs_domain,mapping_status
0,1,South America,Other territories,Manufacturing of crafts and design goods,2002,2.026,Exports,cer7,CER020,C. Visual arts (crafts) / F. Design,provisional_straddle
1,8,South America,Other territories,Books and publishing,2002,3.151,Exports,cer7,CER030,D. Books and press,confirmed
2,9,South America,Other territories,"Music, performing and visual arts",2002,0.006,Exports,cer7,CER040,B. Performance and celebration / C. Visual arts,provisional_straddle


---
## Parte 2 — Questões (10,0 pontos)

A partir daqui, o notebook é de vocês. Boa prova!

---
### Questão 1 — Fundamentos e exploração (2,0 pontos)

A partir de `edges7`, faça:

**a) (0,5 pt)** Construa uma **lista** com os nomes únicos dos domínios (`fcs_domain`) presentes na base.

**b) (0,5 pt)** Construa um **dicionário** que associe cada domínio ao número de linhas (fluxos) correspondentes na base.

**c) (1,0 pt) 🗣️** Usando esse dicionário, identifique qual domínio tem mais fluxos registrados. Na célula de markdown abaixo, escreva **uma frase de negócio** explicando o que esse número representa e uma hipótese para o porquê desse domínio ter tantos registros (dica: pense no número de países que participam desse mercado, não apenas no valor exportado).

In [26]:
unique_fcs_domains = edges7['fcs_domain'].unique().tolist()
print(unique_fcs_domains)

['C. Visual arts (crafts) / F. Design', 'D. Books and press', 'B. Performance and celebration / C. Visual arts', 'E. Audiovisual and interactive media', 'F. Design and creative services', 'A. Cultural and natural heritage']


In [27]:
# Questao 1b — ESCREVA SEU CODIGO AQUI
# Parte 1b: Construa um dicionário que associe cada domínio ao número de linhas (fluxos) correspondentes na base.
domain_counts_dict = edges7['fcs_domain'].value_counts().to_dict()
print(domain_counts_dict)

{'C. Visual arts (crafts) / F. Design': 316124, 'D. Books and press': 245227, 'E. Audiovisual and interactive media': 198292, 'B. Performance and celebration / C. Visual arts': 175540, 'A. Cultural and natural heritage': 56313, 'F. Design and creative services': 34904}


**✍️ Questão 1c — sua resposta em texto aqui:**

O domínio com mais fluxos registrados é 'C. Visual arts (crafts) / F. Design', com 316.124 fluxos, o que pode indicar a alta granularidade e a participação de um vasto número de pequenos produtores e artesãos nesse segmento, gerando um volume elevado de transações individuais.

---
### Questão 2 — Filtragem, ordenação e agregação com pandas (2,5 pontos)

**a) (1,0 pt)** Filtre `edges7` para obter apenas os fluxos em que o **Brasil** é o país exportador (`economy`), em qualquer domínio ou ano.

**b) (1,0 pt)** A partir desse filtro, use `groupby` para calcular o **valor total exportado pelo Brasil, por domínio** (somando todos os anos).

**c) (0,5 pt) 🗣️** Em qual domínio o Brasil mais exporta, em valor total? Responda em texto, citando o número.

In [28]:
# Questao 2a e 2b — ESCREVA SEU CODIGO AQUI

# Parte 2a: Filtrar edges7 para obter apenas os fluxos em que o Brasil é o país exportador.
brazil_exports = edges7[edges7['economy'] == 'Brazil']

# Parte 2b: Calcular o valor total exportado pelo Brasil, por domínio.
total_exported_by_domain = brazil_exports.groupby('fcs_domain')['value_usd_millions'].sum().reset_index()

# Ordenar para facilitar a identificação do domínio com mais exportações (para 2c)
total_exported_by_domain = total_exported_by_domain.sort_values(by='value_usd_millions', ascending=False)

print("Fluxos de exportação do Brasil:")
display(brazil_exports.head())

print("\nValor total exportado pelo Brasil por domínio:")
display(total_exported_by_domain)

Fluxos de exportação do Brasil:


,edge_id,economy,partner,product,year,value_usd_millions,flow,resolution,cer_code,fcs_domain,mapping_status
7844,7845,Brazil,South America,"Audiovisual, multimedia and photography",2002,0.000,Exports,cer7,CER010,E. Audiovisual and interactive media,confirmed
7845,7846,Brazil,South America,Manufacturing of crafts and design goods,2002,50.710,Exports,cer7,CER020,C. Visual arts (crafts) / F. Design,provisional_straddle
7853,7854,Brazil,South America,Books and publishing,2002,5.532,Exports,cer7,CER030,D. Books and press,confirmed
7854,7855,Brazil,South America,"Music, performing and visual arts",2002,1.159,Exports,cer7,CER040,B. Performance and celebration / C. Visual arts,provisional_straddle
7855,7856,Brazil,South America,Architecture,2002,0.000,Exports,cer7,CER050,F. Design and creative services,confirmed



Valor total exportado pelo Brasil por domínio:


,fcs_domain,value_usd_millions
2,C. Visual arts (crafts) / F. Design,31724.509
4,E. Audiovisual and interactive media,3981.583
1,B. Performance and celebration / C. Visual arts,3108.201
3,D. Books and press,1560.670
0,A. Cultural and natural heritage,62.284
5,F. Design and creative services,0.796


**✍️ Questão 2c — sua resposta em texto aqui:**

O domínio em que o Brasil mais exporta, em valor total, é 'C. Visual arts (crafts) / F. Design', com um valor de 31.724,509 milhões de dólares.

---
### Questão 3 — SQL com DuckDB (2,5 pontos)

**a) (1,5 pt)** Escreva uma consulta SQL (via `duckdb.sql(...)`) que retorne, **para o ano de 2023**, o valor total exportado por domínio (`fcs_domain`), ordenado do maior para o menor.

**b) (1,0 pt)** Compare o resultado dessa consulta com o resultado equivalente obtido via `pandas` (`groupby`). Os dois batem? Mostre a comparação no código (não basta afirmar em texto — imprima os dois resultados lado a lado ou calcule a diferença).

In [29]:
# Questao 3a: Consulta SQL para valor total exportado por domínio em 2023
sql_query_3a = """
SELECT
    fcs_domain,
    SUM(value_usd_millions) AS total_exported_2023_sql
FROM
    edges7
WHERE
    year = 2023
GROUP BY
    fcs_domain
ORDER BY
    total_exported_2023_sql DESC
"""

sql_result_2023 = duckdb.sql(sql_query_3a).df()
print("Resultado da consulta SQL (Questão 3a):")
display(sql_result_2023)

# Questao 3b: Comparar com resultado equivalente via pandas

# Filtrar e agrupar com pandas para 2023
pandas_result_2023 = edges7[edges7['year'] == 2023].groupby('fcs_domain')['value_usd_millions'].sum().reset_index()
pandas_result_2023 = pandas_result_2023.rename(columns={'value_usd_millions': 'total_exported_2023_pandas'})
pandas_result_2023 = pandas_result_2023.sort_values(by='total_exported_2023_pandas', ascending=False).reset_index(drop=True)

print("\nResultado via pandas (equivalente à Questão 3a):")
display(pandas_result_2023)

# Comparação dos resultados
comparison_df = pd.merge(sql_result_2023, pandas_result_2023, on='fcs_domain', how='outer')
comparison_df['difference'] = comparison_df['total_exported_2023_sql'] - comparison_df['total_exported_2023_pandas']

print("\nComparação dos resultados SQL vs. Pandas:")
display(comparison_df)

# Verificar se os resultados batem (ignorando pequenas diferenças de floating point, se houver)
if comparison_df['difference'].abs().sum() < 1e-6:
    print("\nOs resultados da consulta SQL e do pandas batem.")
else:
    print("\nHá diferenças entre os resultados da consulta SQL e do pandas.")

Resultado da consulta SQL (Questão 3a):


,fcs_domain,total_exported_2023_sql
0,C. Visual arts (crafts) / F. Design,993887.877
1,E. Audiovisual and interactive media,154384.619
2,B. Performance and celebration / C. Visual arts,48455.694
3,D. Books and press,34507.589
4,A. Cultural and natural heritage,13734.846
5,F. Design and creative services,124.097



Resultado via pandas (equivalente à Questão 3a):


,fcs_domain,total_exported_2023_pandas
0,C. Visual arts (crafts) / F. Design,993887.877
1,E. Audiovisual and interactive media,154384.619
2,B. Performance and celebration / C. Visual arts,48455.694
3,D. Books and press,34507.589
4,A. Cultural and natural heritage,13734.846
5,F. Design and creative services,124.097



Comparação dos resultados SQL vs. Pandas:


,fcs_domain,total_exported_2023_sql,total_exported_2023_pandas,difference
0,A. Cultural and natural heritage,13734.846,13734.846,8.367351e-11
1,B. Performance and celebration / C. Visual arts,48455.694,48455.694,5.020411e-10
2,C. Visual arts (crafts) / F. Design,993887.877,993887.877,3.492460e-10
3,D. Books and press,34507.589,34507.589,4.656613e-10
4,E. Audiovisual and interactive media,154384.619,154384.619,-9.604264e-10
5,F. Design and creative services,124.097,124.097,3.836931e-13



Os resultados da consulta SQL e do pandas batem.


---
### Questão 4 — JOIN e modelagem em estrela (3,0 pontos)

**a) (1,0 pt)** Antes de escrever qualquer `JOIN`, inspecione as colunas de `crosswalk` (`.columns.tolist()`) e identifique, em uma célula de markdown, qual coluna representa a chave de produto (equivalente à coluna `cer_code` em `edges7`) e qual coluna representa o status de confirmação do mapeamento.

**b) (1,0 pt)** Escreva uma consulta SQL com `LEFT JOIN` entre `edges7` e `crosswalk` que retorne, para os **10 maiores fluxos de exportação de qualquer domínio em 2024**, também a informação de status de confirmação do mapeamento de produto.

**c) (1,0 pt) 🗣️** Sem escrever código: descreva, em poucas frases, como você desenharia um **esquema em estrela** para responder a esta pergunta de negócio: *"Qual é o parceiro comercial mais importante de cada economia, em cada domínio?"* — Qual seria a tabela fato? Quais dimensões você usaria? Justifique.

**✍️ Questão 4a — colunas identificadas:**

A coluna que representa a chave de produto, equivalente a `cer_code` em `edges7`, é a coluna `cer` em `crosswalk`.
A coluna que representa o status de confirmação do mapeamento é `mapping_status`.

In [30]:
# Questao 4b: Consulta SQL com LEFT JOIN para os 10 maiores fluxos de exportação em 2024
sql_query_4b = """
SELECT
    e.economy,
    e.partner,
    e.product,
    e.year,
    e.value_usd_millions,
    e.fcs_domain,
    c.status  -- Corrigido de c.mapping_status para c.status
FROM
    edges7 AS e
LEFT JOIN
    crosswalk AS c ON e.cer_code = c.cer
WHERE
    e.year = 2024
ORDER BY
    e.value_usd_millions DESC
LIMIT 10
"""

sql_result_4b = duckdb.sql(sql_query_4b).df()
print("10 maiores fluxos de exportação em 2024 com status de mapeamento:")
display(sql_result_4b)

10 maiores fluxos de exportação em 2024 com status de mapeamento:


,economy,partner,product,year,value_usd_millions,fcs_domain,status
0,G-77 (Group of 77),United States,Manufacturing of crafts and design goods,2024,71596.783,C. Visual arts (crafts) / F. Design,provisional_straddle
1,China,G-77 (Group of 77),Manufacturing of crafts and design goods,2024,61274.620,C. Visual arts (crafts) / F. Design,provisional_straddle
2,China,United States,Manufacturing of crafts and design goods,2024,45191.825,C. Visual arts (crafts) / F. Design,provisional_straddle
3,G-77 (Group of 77),"China, Hong Kong SAR",Manufacturing of crafts and design goods,2024,18707.251,C. Visual arts (crafts) / F. Design,provisional_straddle
4,China,"China, Hong Kong SAR",Manufacturing of crafts and design goods,2024,12057.264,C. Visual arts (crafts) / F. Design,provisional_straddle
5,G-77 (Group of 77),United Arab Emirates,Manufacturing of crafts and design goods,2024,11877.099,C. Visual arts (crafts) / F. Design,provisional_straddle
6,G-77 (Group of 77),Japan,Manufacturing of crafts and design goods,2024,10183.882,C. Visual arts (crafts) / F. Design,provisional_straddle
7,G-77 (Group of 77),United States,"Software, video games and recorded media",2024,9903.889,E. Audiovisual and interactive media,provisional
8,G-77 (Group of 77),United Kingdom,Manufacturing of crafts and design goods,2024,9835.139,C. Visual arts (crafts) / F. Design,provisional_straddle
9,G-77 (Group of 77),South America,Manufacturing of crafts and design goods,2024,8567.727,C. Visual arts (crafts) / F. Design,provisional_straddle


**✍️ Questão 4c — sua resposta em texto aqui:**

*(substitua este texto pela sua resposta)*

Para a pergunta de negócio _'Qual é o parceiro comercial mais importante de cada economia, em cada domínio?'_:

*   **Tabela Fato:** `Fato_Comercio`
    *   **Métricas:** `valor_exportado_usd` (agregaria `value_usd_millions`).
    *   **Chaves Estrangeiras:** `id_economia`, `id_parceiro`, `id_dominio`, `id_tempo`.
    *   **Justificativa:** Esta tabela registraria os eventos de comércio (fluxos de exportação) e o valor associado, sendo o centro para analisar o volume comercial.

*   **Tabelas de Dimensão:**
    *   `Dim_Economia`: Incluiria `id_economia`, `nome_economia` (proveniente de `economy` na `edges7`).
        *   **Justificativa:** Permite filtrar e agrupar dados por cada economia.
    *   `Dim_Parceiro`: Incluiria `id_parceiro`, `nome_parceiro` (proveniente de `partner` na `edges7`).
        *   **Justificativa:** Permite identificar e analisar o desempenho de cada parceiro comercial.
    *   `Dim_Dominio_Produto`: Incluiria `id_dominio`, `fcs_domain`, `cer_code`, `product`, `mapping_status` (unindo informações de `edges7` e `crosswalk`).
        *   **Justificativa:** Permite analisar o comércio dentro de diferentes domínios e produtos, usando as informações de mapeamento.
    *   `Dim_Tempo`: Incluiria `id_tempo`, `ano` (proveniente de `year` na `edges7`).
        *   **Justificativa:** Permite a análise temporal dos fluxos comerciais.

---
## ✅ Checklist final antes de entregar

- [ ] Todas as células rodam em sequência, do início ao fim, sem erro (teste com *Ambiente de execução → Executar tudo*).
- [ ] Todas as perguntas 🗣️ foram respondidas em texto, não só em código.
- [ ] Baixei o notebook (`.ipynb`) e subi para um repositório público no GitHub.
- [ ] Criei um `README.md` no repositório (modelo fornecido pelo professor).
- [ ] Enviei o link do repositório no local indicado, dentro do prazo.

**Boa prova!**